In [0]:
from pyspark.sql import functions as F

def process_orders():
    orders_schema = "order_id STRING, order_timestamp timestamp, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

    query = (spark.readStream
                    .table("workspace.bookstore_eng_pro.bronze")
                    .filter("topic = 'orders'")
                    .select(F.from_json(F.unbase64(F.col("value")).cast("string"), orders_schema).alias("v"))
                    .select("v.*")
                    .withColumn("load_timestamp", F.current_timestamp())
                    .writeStream
                    .option("checkpointLocation", "/Volumes/workspace/bookstore_eng_pro/checkpoints/orders_checkpoint")
                    .trigger(availableNow=True)
                    .table("workspace.bookstore_eng_pro.orders")
    )
    query.awaitTermination()

In [0]:
process_orders()